# Project Summary — Early Warning for Long Administrative Cases

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

**Question:** Can early information from an administrative process help us identify which cases are likely to take
longer than usual?

This notebook contains **no code**. It summarises notebooks 01–12; every number below comes from their saved outputs.
Follow the links for the details.

## 1. Data and Target

- **Data:** BPI Challenge 2015, Municipality 1: building-permit cases from **one Dutch municipality** (52,217 events,
  1,199 cases). This is public research data, **not SPAR data**.
- **Completed cases:** cases with a recorded `case:endDate` (956 cases).
- **Recorded activity span:** days from the first to the last recorded event of a case. It is not a verified
  administrative completion time.
- **Long case:** a span of **more than 63 days**. The 63 days come from the exploratory analysis (the median span of the
  completed cases in [01_EDA](01_EDA.ipynb)); it is not an external deadline.

## 2. How the Project Developed

| Step | Notebooks | What I tried | What I learned |
|---|---|---|---|
| 1 | [01](01_EDA.ipynb) | Load and explore the event log | Key columns complete; spans vary a lot (median 63 days, long tail) |
| 2 | [02](02_linear_regression_review.ipynb), [03](03_decision_tree_review.ipynb), [04](04_random_forest_review.ipynb), [05](05_gradient_boosting_review.ipynb) | Predict the span from **whole-case** features | MAE fell from 71.98 (mean baseline) to 51.19 (Gradient Boosting), but these features only exist after a case ends, so they are learning exercises, **not** early warnings |
| 3 | [06](06_early_warning_features_review.ipynb) | Use only the first 5 events | Weak signal: accuracy 0.55 vs 0.51 baseline |
| 4 | [07](07_14_day_early_warning_review.ipynb), [08](08_14_day_activity_features_review.ipynb) | Use a fixed **day-14** snapshot: counts, then which activities were recorded | Counts alone: no signal. With activities: ROC-AUC 0.53 → 0.59, a small exploratory gain |
| 5 | [09](09_day14_regression_review.ipynb) | Predict the span in days from day-14 features | No model beat the median baseline (MAE 70.75) |
| 6 | [10](10_day30_early_warning_review.ipynb) | Use a later **day-30** snapshot with 5-fold cross-validation | Modest signal: Logistic Regression ROC-AUC 0.702 (cross-validation), 0.659 (test) |
| 7 | [11](11_day30_temporal_validation_review.ipynb) | Train on **older** cases, test on **newer** cases | Similar ranking (ROC-AUC 0.678) |
| 8 | [12](12_day30_model_comparison_review.ipynb) | Compare Random Forest and Gradient Boosting at day 30 | No consistent gain over Logistic Regression |

The experiments that did not beat their baseline are part of the result: they showed that the first days of a case
contain little information, and they led to the later day-30 snapshot.

## 3. Main Day-30 Comparison

**ROC-AUC** measures how well a model ranks long cases above not-long cases: 0.50 is no better than chance.

| ROC-AUC | Cross-validation | Random test | Temporal test |
|---|---|---|---|
| Majority baseline | 0.500 | 0.500 | 0.500 |
| Logistic Regression | 0.702 | 0.659 | 0.678 |
| Random Forest | 0.701 | 0.745 | 0.674 |
| Gradient Boosting | 0.684 | 0.723 | 0.624 |

- **Cross-validation:** the model is trained and checked on five different portions of the training cases.
- **Random test:** notebook 07's original random test cases (161 at day 30).
- **Temporal test:** the newest cases (168), with the model trained only on older cases.

Random Forest was strongest on the random test, but practically equal to Logistic Regression in cross-validation and on the
temporal test. Because the scores are similar, the simpler **Logistic Regression** is kept as the main candidate.

## 4. What the Baseline Means

The **majority baseline** predicts "long" for every case, because long cases are the more common class at day 30. It
therefore finds **every** long case (recall 1.00), but it cannot tell cases apart (ROC-AUC 0.50).

That is why recall and F1 alone can look good for a useless model. A real warning model must do better at **ranking**
cases and give warnings that are more often correct:

| Test | Model | Cases flagged | Warnings correct (precision) | Long cases found (recall) |
|---|---|---|---|---|
| Random test (161 cases) | Majority baseline | 161 | 59.0% | 100% |
| Random test (161 cases) | Logistic Regression | 98 | 69.4% | 71.6% |
| Temporal test (168 cases) | Majority baseline | 168 | 59.5% | 100% |
| Temporal test (168 cases) | Logistic Regression | 71 | 77.5% | 55% |

Logistic Regression's warnings are more often correct than random flagging, but it misses many long cases, especially on
the newer cases.

## 5. Limitations

- The target is the **recorded activity span**; for many completed cases the last recorded event is after the recorded end
  date, and for some it is before it.
- Only **historically completed** cases are included.
- Timestamps are assumed to show when information was available; the leakage checks verify the feature code, not this assumption.
- The same test cases were used in several notebooks, so the later results are **exploratory**.
- **One municipality**; a similar ROC-AUC on newer cases does not prove that the model generalises or would be useful in practice.
- Forecasting the number of new cases per month was **assessed for feasibility only** and was not modelled.

## 6. Connection to SPAR

This project shows a **method worth evaluating on SPAR's own historical data**: build features from what is known at a
fixed early moment, compare them with a simple baseline, and check the result on newer cases. It is **not** a model that
has been validated for SPAR or Montenegro, and its results cannot be transferred directly.